# EDA: GTZAN Music Genre Dataset

In [ ]:
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
import IPython.display
import os
import json
import pathlib
import pandas as pd
import yaml


In [ ]:
with open('../configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Genres:", config['data']['genres'])
print("Sample rate:", config['data']['sample_rate'], "Hz")
print("Duration:", config['data']['duration'], "s")
print("Processed path:", config['data']['processed_dir'])


In [ ]:
raw_dir = config['data']['raw_dir']
genres = config['data']['genres']
counts = {}
for g in genres:
    p = os.path.join(raw_dir, g)
    if os.path.exists(p):
        counts[g] = len([f for f in os.listdir(p) if f.endswith('.wav')])
    else:
        counts[g] = 0

plt.figure(figsize=(10, 6))
plt.barh(list(counts.keys()), list(counts.values()), color='skyblue')
plt.title('Class Balance')
plt.xlabel('Number of Files')
plt.ylabel('Genre')
plt.show()
print("Counts per genre:", counts)


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 6))
axes = axes.flatten()
for i, g in enumerate(genres):
    folder = os.path.join(raw_dir, g)
    if not os.path.exists(folder): continue
    files = [x for x in os.listdir(folder) if x.endswith('.wav')]
    if not files: continue
    y, sr = librosa.load(os.path.join(folder, files[0]), sr=config['data']['sample_rate'])
    librosa.display.waveshow(y, sr=sr, ax=axes[i])
    axes[i].set_title(g.capitalize())
plt.tight_layout()
plt.show()


In [ ]:
processed_dir = config['data']['processed_dir']
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()
for i, g in enumerate(genres):
    folder = os.path.join(processed_dir, g)
    if not os.path.exists(folder): continue
    files = [x for x in os.listdir(folder) if x.endswith('.npy')]
    if not files: continue
    mel = np.load(os.path.join(folder, files[0]))
    img = librosa.display.specshow(mel, sr=config['data']['sample_rate'], hop_length=config['data']['hop_length'], x_axis='time', y_axis='mel', ax=axes[i], cmap='magma')
    axes[i].set_title(g.capitalize())
plt.tight_layout()
plt.show()


In [ ]:
stats = []
for g in genres:
    folder = os.path.join(processed_dir, g)
    if not os.path.exists(folder): continue
    files = [x for x in os.listdir(folder) if x.endswith('.npy')][:10]
    for f in files:
        mel = np.load(os.path.join(folder, f))
        stats.append({
            'genre': g,
            'mean_energy': mel.mean(),
            'std_energy': mel.std(),
            'dynamic_range': mel.max() - mel.min()
        })
df = pd.DataFrame(stats)
if not df.empty:
    print(df.groupby('genre').mean())


### Observations
- Classical shows strong harmonic stacks and lower overall average energy compared to metal.
- Hip-hop shows concentrated bass energy at lower frequencies with clear rhythmic pulses.
- Metal is characterized by high-energy, broadband noise spanning across most mel bands.
- Disco and Pop display strong, regular beats and bright high-frequency synth components.


In [ ]:
data_dir = os.path.dirname(processed_dir)
files_to_check = ['train_split.json', 'val_split.json', 'test_split.json']
for fname in files_to_check:
    fpath = os.path.join('..', 'data', fname)
    if os.path.exists(fpath):
        with open(fpath, 'r') as f: data = json.load(f)
        print(f"{fname}: {len(data['files'])} samples")
    else:
        print(f"{fname} MISSING")
